In [ ]:
%load_ext rpy2.ipython

import os

import matplotlib.pyplot as plt
import nivapy3 as nivapy
import pandas as pd

plt.style.use("ggplot")

In [ ]:
# %%R
# install.packages("EGRET",repos = "https://cloud.r-project.org")

# Flow-normalised loads and concentrations

Initial notebook for developing workflow.

## 1. User input

In [ ]:
par = "TOTN_ug/l"
stn_code = "BUSEDRA"

## 2. Read water chemistry and flow

In [ ]:
wc_xls = r"../data/oslofjord_rid_chem.xlsx"
q_xls = r"../data/oslofjord_rid_flows.xlsx"

wc_df = (
    pd.read_excel(wc_xls)
    .rename(columns={"sample_date": "date"})
    .query("station_code == @stn_code")[["date", par]]
    .set_index("date")
    .resample("D")
    .mean()
    .dropna(subset=par)
)
q_df = (
    pd.read_excel(q_xls)[["date", f"{stn_code}_m3/s"]]
    .set_index("date")
    .resample("D")
    .mean()
    .rename(columns={f"{stn_code}_m3/s": "flow_m3/s"})
)

## 3. Estimate OSPAR annual fluxes

In [ ]:
osp_df = (
    nivapy.stats.estimate_fluxes(
        q_df,
        wc_df,
        base_freq="D",
        agg_freq="YE",
        method="ospar_annual",
    )
    .dropna(how="all")
    .reset_index()
)

# Convert to tonnes
osp_df[f"OSPAR_{par.split('_')[0]}_tonn"] = osp_df[f"{par.split('_')[0]}_kg"] / 1000
cols = [col for col in osp_df.columns if col.endswith("_kg")]
del osp_df[f"{par.split('_')[0]}_kg"]
osp_df[f"OSPAR_{par.split('_')[0]}_tonn"] = osp_df[
    f"OSPAR_{par.split('_')[0]}_tonn"
].round(-1)

osp_df.head()

## 4. Estimate fluxes using EGRET

### 4.1. Prepare data

In [ ]:
# Filter chemistry to overlap with flow record
q_start = q_df.index.min()
q_end = q_df.index.max()

wc_df = wc_df.loc[(wc_df.index >= q_start) & (wc_df.index <= q_end)].copy()

# EGRET sample file
wc_df = wc_df.reset_index().rename(columns={par: "TN"})
wc_df["date"] = wc_df["date"].dt.strftime("%Y-%m-%d")
wc_df["remarkCode"] = ""
wc_df = wc_df[["date", "remarkCode", "TN"]]
wc_df["TN"] = wc_df["TN"] / 1000  # mg/L
wc_df.to_csv("egret_samples.csv", index=False)

# EGRET daily flow file
q_df = q_df.reset_index().rename(columns={"flow_m3/s": "Q"})
q_offset = q_df.loc[q_df["Q"] > 0, "Q"].min()
print(f"Adding flow offset of {q_offset:.4f} m3/s to avoid days with zero flow.")
q_df["Q"] = q_df["Q"] + q_offset
q_df["date"] = q_df["date"].dt.strftime("%Y-%m-%d")
q_df = q_df[["date", "Q"]]
q_df.to_csv("egret_daily.csv", index=False)

### 4.2. Fit model

In [ ]:
%%R

library(EGRET)

# Read EGRET input files
Sample <- readUserSample(
  filePath = ".",
  fileName = "egret_samples.csv",
  separator = ","
)

Daily <- readUserDaily(
  filePath = ".",
  fileName = "egret_daily.csv",
  separator = ",",
  qUnit = 2  # m3/s
)

INFO <- data.frame(
  shortName      = "Station",
  paramShortName = "TN",
  constitAbbrev  = "TN",
  param.units    = "mg/L",
  drainSqKm      = 42000,
  paStart        = 1,
  paLong         = 12
)

# Create EGRET object
eList <- as.egret(
  INFO   = INFO,
  Daily  = Daily,
  Sample = Sample
)

# Fit WRTDS model
eList <- modelEstimation(eList)

# Annual results
annual_results <- tableResults(eList)

# Save
write.csv(
  annual_results,
  "egret_results.csv",
  row.names = FALSE
)

In [ ]:
# Read results
egret_df = pd.read_csv("egret_results.csv")
egret_df.head()

## 5. Estimate fluxes using GAM

In [ ]:
%%R

library(mgcv)
library(dplyr)
library(MASS)

# Read data
sample_df <- read.csv(
  "egret_samples.csv",
  stringsAsFactors = FALSE
)
daily_df <- read.csv(
  "egret_daily.csv",
  stringsAsFactors = FALSE
)
sample_df$date <- as.Date(sample_df$date)
daily_df$date  <- as.Date(daily_df$date)

# Prepare data
daily_df <- daily_df %>%
  filter(
    !is.na(Q),
    Q > 0
  ) %>%
  mutate(
    year = as.integer(format(date, "%Y")),
    doy = as.integer(format(date, "%j")),
    dec_year = year + (doy - 0.5) / 365.25
  )
sample_df <- sample_df %>%
  left_join(
    daily_df %>%
      dplyr::select(date, Q),
    by = "date"
  ) %>%
  mutate(
    year = as.integer(format(date, "%Y")),
    doy = as.integer(format(date, "%j")),
    dec_year = year + (doy - 0.5) / 365.25
  ) %>%
  filter(
    !is.na(TN),
    !is.na(Q),
    TN > 0,
    Q > 0
  )
cat("Samples used:", nrow(sample_df), "\n")

# Fit GAM
gam_fit <- gam(
  log(TN) ~
    s(log(Q), k = 10) +
    s(dec_year, k = 20) +
    s(doy, bs = "cc", k = 20) +
    ti(
      log(Q),
      doy,
      bs = c("tp", "cc"),
      k = c(10, 10)
    ),
  data = sample_df,
  method = "REML"
)

# Model summary
print(summary(gam_fit))
plot(
  gam_fit,
  pages = 1,
  shade = TRUE,
  residuals = TRUE
)

# Predictions at sample dates
sample_df$pred_log <- predict(
  gam_fit,
  newdata = sample_df
)
resid_var <- var(
  residuals(gam_fit)
)
sample_df$pred_TN <- exp(
  sample_df$pred_log +
  0.5 * resid_var
)

# Daily predictions
daily_df$pred_log_c <- predict(
  gam_fit,
  newdata = daily_df
)
daily_df$pred_c <- exp(
  daily_df$pred_log_c +
  0.5 * resid_var
)

# Daily loads
# mg/L × m3/s × 86.4 = kg/day
daily_df$load_kg_day <-
  daily_df$pred_c *
  daily_df$Q *
  86.4

# Annual estimates
annual_loads <- daily_df %>%
  group_by(year) %>%
  summarise(
    load_kg = sum(
      load_kg_day,
      na.rm = TRUE
    ),
    .groups = "drop"
  ) %>%
  mutate(
    load_tonnes = load_kg / 1000
  )

# Bayesian posterior simulation
n_sim <- 1000
cat(
  "Running",
  n_sim,
  "posterior simulations...\n"
)
Xp <- predict(
  gam_fit,
  newdata = daily_df,
  type = "lpmatrix"
)
beta_hat <- coef(gam_fit)
Vp <- vcov(
  gam_fit,
  unconditional = TRUE
)
beta_sim <- MASS::mvrnorm(
  n = n_sim,
  mu = beta_hat,
  Sigma = Vp
)
n_years <- nrow(annual_loads)
annual_sim <- matrix(
  NA_real_,
  nrow = n_years,
  ncol = n_sim
)

for (i in seq_len(n_sim)) {
  pred_log_sim <- as.vector(
    Xp %*% beta_sim[i, ]
  )
  pred_c_sim <- exp(
    pred_log_sim +
    0.5 * resid_var
  )
  annual_i <- data.frame(
    year = daily_df$year,
    load_kg =
      pred_c_sim *
      daily_df$Q *
      86.4
  ) %>%
    group_by(year) %>%
    summarise(
      load_kg = sum(
        load_kg,
        na.rm = TRUE
      ),
      .groups = "drop"
    )
  annual_sim[, i] <- annual_i$load_kg
}

# 95% credible intervals
annual_loads$lower_kg <- apply(
  annual_sim,
  1,
  quantile,
  probs = 0.025,
  na.rm = TRUE
)
annual_loads$upper_kg <- apply(
  annual_sim,
  1,
  quantile,
  probs = 0.975,
  na.rm = TRUE
)
annual_loads <- annual_loads %>%
  mutate(
    lower_tonnes = lower_kg / 1000,
    upper_tonnes = upper_kg / 1000
  )

# Annual uncertainty (%)
annual_loads <- annual_loads %>%
  mutate(
    rel_uncertainty_pct =
      100 *
      (upper_tonnes - lower_tonnes) /
      (2 * load_tonnes)
  )

# Save
write.csv(
  annual_loads,
  "gam_results.csv",
  row.names = FALSE
)
cat("GAM analysis complete.\n")

In [ ]:
# Read results
gam_df = pd.read_csv("gam_results.csv")[
    ["year", "load_tonnes", "lower_tonnes", "upper_tonnes"]
]
gam_df.columns = [
    "year",
    f"GAM_{par.split('_')[0]}_tonn",
    f"lower_GAM_{par.split('_')[0]}_tonn",
    f"upper_GAM_{par.split('_')[0]}_tonn",
]
gam_df.head()

## 6. Compare

In [ ]:
# EGRET
egret_df["Flux [10^6kg/yr]"] = 1000 * egret_df["Flux [10^6kg/yr]"]
egret_df["FN Flux [10^6kg/yr]"] = 1000 * egret_df["FN Flux [10^6kg/yr]"]
egret_df = egret_df.rename(
    columns={
        "Year": "year",
        "Flux [10^6kg/yr]": f"EGRET_{par.split('_')[0]}_tonn",
        "FN Flux [10^6kg/yr]": f"EGRET_FN_{par.split('_')[0]}_tonn",
    }
)
egret_df = egret_df[
    ["year", f"EGRET_{par.split('_')[0]}_tonn", f"EGRET_FN_{par.split('_')[0]}_tonn"]
]
egret_df["year"] = egret_df["year"] - 1  # EGRET uses year end

# Join
df = pd.merge(osp_df, egret_df, on="year", how="inner")
df = pd.merge(df, gam_df, on="year", how="inner").set_index("year").round(-1)

# Delete temp. files
os.remove(r"egret_daily.csv")
os.remove(r"egret_samples.csv")
os.remove(r"egret_results.csv")
os.remove(r"gam_results.csv")

df.head()

### 6.1. Time series

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 6), sharex=True, sharey=True)
axes=axes.flatten()

df[f"OSPAR_{par.split('_')[0]}_tonn"].plot(c='k', marker="o", ax=axes[0])
axes[0].legend()
axes[0].set_xlabel('')
axes[0].set_ylabel(f'{par.split('_')[0]} load (tonnes/year)')

df[f"EGRET_{par.split('_')[0]}_tonn"].plot(c='b', marker="o", ax=axes[1])
axes[1].legend()
axes[1].set_xlabel('')
axes[1].set_ylabel(f'{par.split('_')[0]} load (tonnes/year)')

df[f"EGRET_FN_{par.split('_')[0]}_tonn"].plot(c='g', marker="o", ax=axes[2])
axes[2].legend()
axes[2].set_xlabel('')
axes[2].set_ylabel(f'{par.split('_')[0]} load (tonnes/year)')

df[f"GAM_{par.split('_')[0]}_tonn"].plot(c='r', marker="o", ax=axes[3])
axes[3].fill_between(
    df.index,
    df[f"lower_GAM_{par.split('_')[0]}_tonn"],
    df[f"upper_GAM_{par.split('_')[0]}_tonn"],
    alpha=0.2,
    color="r",  # adjust if GAM is a different colour
    label="GAM 95% CI"
)
axes[3].legend()
axes[3].set_xlabel('')
axes[3].set_ylabel(f'{par.split('_')[0]} load (tonnes/year)')

plt.tight_layout()

### 6.2. Trends

In [ ]:
print(f"Results for {par.split('_')[0]} (in tonnes) at {stn_code}:")
cols = [
    col
    for col in df.columns
    if not col.startswith(("lower", "upper"))
]
for col in cols:
    mk_res = nivapy.stats.mk_test(df, col, alpha=0.05)
    res_df, sen_df = nivapy.stats.sens_slope(df, col, index_col=df.index, alpha=0.05)
    trend = mk_res.loc["trend", "value"]
    pval = mk_res.loc["p", "value"]
    sslp = res_df.loc["sslp", "value"]
    icpt = res_df.loc["icpt", "value"]
    chng = (sslp * df.index.max() + icpt) - (sslp * df.index.min() + icpt)
    pct_chng = 100 * chng / (sslp * df.index.min() + icpt)

    print(f"  {col}")
    print(f"    Trend: {trend} (p = {pval:.4f})")
    print(f"    {pct_chng:.1f} % over {df.index.max() - df.index.min()} years.")